In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings("ignore")
import pickle
import os

In [2]:
lipidProfile = pd.read_csv(r"C:\Users\direk\Disease_risk_predictor_-3\SYSTEM\dataset\lipid_profile.csv")
lipidProfile = lipidProfile.drop("risk_label", axis=1, errors="ignore")
lipidProfile.head()

,total_cholesterol,hdl,ldl,triglycerides,vldl,heart_disease_risk,stroke_risk
0,173.50,52.62,86.22,168.57,33.08,5.55,15.13
1,168.40,44.64,101.45,117.78,24.09,25.78,6.37
2,149.66,47.59,84.83,60.83,13.60,16.55,15.18
3,202.47,54.41,111.20,208.12,41.52,41.93,35.88
4,184.14,62.49,95.90,127.56,25.54,5.71,9.48


In [3]:
def heart_risk(row):
    score = 0
    
    if row["total_cholesterol"] > 280:
        return "Critical"
    
    if row["total_cholesterol"] > 200:
        score += 1
    if row["ldl"] > 130:
        score += 1
    if row["hdl"] < 40:
        score += 1
    if row["triglycerides"] > 150:
        score += 1
    
    if score >= 3:
        return "High"
    elif score == 2:
        return "Medium"
    else:
        return "Low"


def stroke_risk(row):
    score = 0

    if row["ldl"] > 190 or row["total_cholesterol"] > 300:
        return "Critical"

    if row["ldl"] > 160:
        score += 2
    if row["triglycerides"] > 150:
        score += 1
    if row["total_cholesterol"] > 240:
        score += 2
    
    if score >= 3:
        return "High"
    elif score >= 1:
        return "Medium"
    else:
        return "Low"

In [4]:
lipidProfile["heart_risk"] = lipidProfile.apply(heart_risk, axis=1)
lipidProfile["stroke_risk"] = lipidProfile.apply(stroke_risk, axis=1)

In [5]:
LABEL_MAPPING = {
    "Low": "low",
    "Normal": "moderate",
    "Medium": "moderate",
    "High": "high",
    "Critical": "critical"
}

NUM_MAPPING = {
    "low": 0,
    "moderate": 1,
    "high": 2,
    "critical": 3
}

In [6]:
lipidProfile["heart_risk"] = lipidProfile["heart_risk"].map(LABEL_MAPPING)
lipidProfile["stroke_risk"] = lipidProfile["stroke_risk"].map(LABEL_MAPPING)
lipidProfile["heart_risk_num"] = lipidProfile["heart_risk"].map(NUM_MAPPING)
lipidProfile["stroke_risk_num"] = lipidProfile["stroke_risk"].map(NUM_MAPPING)
lipidProfile.head()

,total_cholesterol,hdl,ldl,triglycerides,vldl,heart_disease_risk,stroke_risk,heart_risk,heart_risk_num,stroke_risk_num
0,173.50,52.62,86.22,168.57,33.08,5.55,moderate,low,0,1
1,168.40,44.64,101.45,117.78,24.09,25.78,low,low,0,0
2,149.66,47.59,84.83,60.83,13.60,16.55,low,low,0,0
3,202.47,54.41,111.20,208.12,41.52,41.93,moderate,moderate,1,1
4,184.14,62.49,95.90,127.56,25.54,5.71,low,low,0,0


In [7]:
"""Preparing data for ML prediction"""
feature_cols = ["total_cholesterol","hdl","ldl","triglycerides","vldl"]
X = lipidProfile[feature_cols]
y_heart = lipidProfile["heart_risk_num"]
y_stroke = lipidProfile["stroke_risk_num"]

In [8]:
"""train test split""" 
X_train, X_test, y_train_heart, y_test_heart = train_test_split(X, y_heart, test_size=0.2, random_state=31, stratify=y_heart)
_, _, y_train_stroke, y_test_stroke = train_test_split(X, y_stroke, test_size=0.2, random_state=31, stratify=y_heart)

In [9]:
"""Auto detect classes and print report - works for any number of classes"""
CLASS_NAMES = {0: "low", 1: "moderate", 2: "high", 3: "critical"}

def print_report(y_test, y_pred, model_name):
    # automatically finds which classes exist in test + predictions
    existing_labels = sorted(np.unique(np.concatenate([y_test, y_pred])))
    existing_names = [CLASS_NAMES[i] for i in existing_labels]

    print("=" * 40)
    print(f"{model_name} MODEL ACCURACY")
    print("=" * 40)
    print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
    print()
    print("=" * 40)
    print(f"{model_name} CLASSIFICATION REPORT")
    print("=" * 40)
    print(classification_report(
        y_test, y_pred,
        labels=existing_labels,
        target_names=existing_names
    ))


In [10]:
model_heart = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,           
    colsample_bytree=0.8,    
    gamma=0.1,               
    use_label_encoder=False,
    eval_metric="mlogloss",
    random_state=31
)
model_heart.fit(X_train, y_train_heart)
y_predict_heart = model_heart.predict(X_test)
print_report(y_test_heart, y_predict_heart, "HEART DISEASE")

HEART DISEASE MODEL ACCURACY
Accuracy: 99.00%

HEART DISEASE CLASSIFICATION REPORT
              precision    recall  f1-score   support

         low       1.00      1.00      1.00        71
    moderate       0.96      1.00      0.98        24
        high       1.00      0.80      0.89         5

    accuracy                           0.99       100
   macro avg       0.99      0.93      0.96       100
weighted avg       0.99      0.99      0.99       100



In [11]:
model_stroke = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,           
    colsample_bytree=0.8,    
    gamma=0.1,               
    use_label_encoder=False,
    eval_metric="mlogloss",
    random_state=31
)
model_stroke.fit(X_train, y_train_stroke)
y_predict_stroke = model_stroke.predict(X_test)
print_report(y_test_stroke, y_predict_stroke, "STROKE")

STROKE MODEL ACCURACY
Accuracy: 96.00%

STROKE CLASSIFICATION REPORT
              precision    recall  f1-score   support

         low       0.97      0.97      0.97        70
    moderate       0.92      0.92      0.92        26
        high       1.00      1.00      1.00         4

    accuracy                           0.96       100
   macro avg       0.96      0.96      0.96       100
weighted avg       0.96      0.96      0.96       100



In [12]:
"""Saving the models as pkl file"""
save_path = r"C:\Users\direk\Disease_risk_predictor_-3\ml_models\xgboost"
os.makedirs(save_path, exist_ok=True)
with open(os.path.join(save_path, "heart.pkl"), "wb") as f:
    pickle.dump(model_heart, f)
    print("heart.pkl is saved successfully")
with open(os.path.join(save_path, "stroke.pkl"), "wb") as f:
    pickle.dump(model_stroke, f)
    print("stroke.pkl is saved successfully")

heart.pkl is saved successfully
stroke.pkl is saved successfully
